# Scratch: Checking the Data Sources Before Building the Subset

These are quick checks made before writing `notebooks/Milestone-3/01_data_preparation.ipynb`: where each
source's files are, how they're stored, and how they're labelled. Nothing here is used by the pipeline.

In [1]:
import json
import tarfile
import zipfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, HfFileSystem, hf_hub_download

api, fs = HfApi(), HfFileSystem()

/tmp/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Kathbath on Hugging Face (gated; needs the terms accepted on the dataset page)

In [2]:
files = [s.rfilename for s in api.dataset_info("ai4bharat/Kathbath").siblings]
pd.Series([f.split("/")[0] + "/" + f.split("/")[1].split("-")[0] for f in files
           if f.startswith(("hindi/", "tamil/", "telugu/"))]).value_counts()

tamil/train     40
telugu/train    33
hindi/train     32
hindi/valid      2
tamil/valid      2
telugu/valid     2
Name: count, dtype: int64

Only `train` and `valid` exist, so **Kathbath's test speakers aren't on Hugging Face.**

In [3]:
parquet = pq.ParquetFile(fs.open("datasets/ai4bharat/Kathbath/hindi/train-00000-of-00032.parquet"))
print(parquet.schema_arrow)
print(parquet.metadata.num_rows, "rows in", parquet.metadata.num_row_groups, "row groups")
first = parquet.read_row_group(0, columns=["audio_filepath"]).column("audio_filepath")[0].as_py()
print(first["path"], first["bytes"][:4])

fname: string
text: string
audio_filepath: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
lang: string
duration: double
gender: string
speaker_id: int64
-- schema metadata --
huggingface: '{"info": {"features": {"fname": {"dtype": "string", "_type"' + 298
2868 rows in 29 row groups


844424930703439-329-f.flac b'fLaC'


The audio is stored as 16 kHz FLAC (`fLaC`), so `soundfile` reads it directly.

## 2. Kathbath Test Speakers: the Official Archive

In [4]:
import subprocess
for name in ["testunk_audio.tar", "testkn_audio.tar", "valid_audio.tar", "train_audio.tar"]:
    head = subprocess.run(["curl", "-sI", f"https://objectstore.e2enetworks.net/indic-superb/kathbath/clean/{name}"],
                          capture_output=True, text=True).stdout
    size = [l for l in head.splitlines() if l.lower().startswith("content-length")]
    print(f"{name:20s} {int(size[0].split()[1]) / 1e9:6.1f} GB")

testunk_audio.tar       2.2 GB


testkn_audio.tar        3.6 GB


valid_audio.tar         3.6 GB


train_audio.tar        91.1 GB


In [5]:
with tarfile.open("../../../data/raw/kathbath/testunk_audio.tar") as tar:
    names = [m.name for m in tar if m.isfile()]
print(len(names), "files, for example", names[0])
pd.Series([n.split("/")[1] for n in names]).value_counts().loc[["hindi", "tamil", "telugu"]]

19672 files, for example kb_data_clean_m4a/punjabi/test/audio/844424932647549-438-f.m4a


hindi     1929
tamil     1642
telugu    1492
Name: count, dtype: int64

`testunk_audio.tar` (2.2 GB) holds the unseen test speakers as m4a, so it's the only archive we need.

## 3. Our Generated Set: Per-Clip Manifest

In [6]:
repo = "datasets/TarikaHedaoo/dsai-group3-fake-speech"
line = fs.open(f"{repo}/manifests/indicf5/hi/hi_0099_indicf5.jsonl").read().decode().splitlines()[0]
json.loads(line)

{'clip_id': 'hi_0001_indicf5',
 'sent_id': 'hi_0001',
 'lang': 'hi',
 'generator': 'indicf5',
 'split': 'test',
 'kb_file': '844424932563142-769-m.m4a',
 'kb_speaker': '769',
 'kb_gender': 'm',
 'n_words': 11,
 'text': 'डियोड्रेंट में कई तरह के सिंथेटिक और रसायन मौजूद होते हैं',
 'voice_id': 'ivr-S4257196900365559',
 'voice_gender': 'm',
 'ref_text': 'क्योंकि इससे मैंने अनेक सी चीज़ें मुझे इससे जानने को मिली मैंने इसका बहुत ही अच्छी तरीके से उपयोग किया',
 'duration_s': 3.381,
 'peak': 0.5135,
 'rms': 0.10001,
 'clipped_frac': 0.0,
 'flag_silent': False,
 'flag_short': False,
 'label': 'fake',
 'path': 'raw/indicf5/hi/hi_0001_indicf5.flac',
 'sha1': 'a0e9714b6aff73d54f5c28e8c230171fa51b735e',
 'sample_rate': 24000,
 'model_id': 'ai4bharat/IndicF5',
 'seed': 42,
 'gen_seconds': 14.08,
 'created_utc': '2026-10-28T19:17:16+00:00',
 'error': None}

Each fake clip has `kb_file`, the Kathbath test recording of the same sentence. That gives a real clip with the same words.

## 4. ASVspoof 2019 LA on Hugging Face

In [7]:
files = [s.rfilename for s in api.dataset_info("SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA").siblings]
print([f for f in files if not f.startswith("data/test-")])
print(pq.ParquetFile(fs.open("datasets/SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA/data/test-00000-of-00009.parquet")).schema_arrow)

['.gitattributes', '.gitignore', 'LICENSE.txt', 'README.md', 'build_parquet.py', 'data/labels.parquet', 'eval.yaml', 'protocols/ASVspoof2019.LA.cm.eval.trl.txt', 'submissions/README.md', 'submissions/aasist-l.yaml', 'submissions/aasist.yaml', 'submissions/aasist3.yaml', 'submissions/amsdf.yaml', 'submissions/clad.yaml', 'submissions/codecfake-w2v2-aasist-codec.yaml', 'submissions/codecfake-w2v2-aasist-cotrain.yaml', 'submissions/df-arena-1b.yaml', 'submissions/df-arena-500m.yaml', 'submissions/dpda-ga-aasist.yaml', 'submissions/dpda-ga-mamba.yaml', 'submissions/dpda-ga-tcm.yaml', 'submissions/fake-mamba.yaml', 'submissions/forensics-xlsr-wild.yaml', 'submissions/holiantispoof.yaml', 'submissions/m1.yaml', 'submissions/nes2net.yaml', 'submissions/random-baseline.yaml', 'submissions/rawbmamba.yaml', 'submissions/rawtfnet.yaml', 'submissions/res2tcnguard.yaml', 'submissions/rescapsguard.yaml', 'submissions/results_template.yaml', 'submissions/sonar-full.yaml', 'submissions/spectra-0.yaml'

path: string
audio: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
label: int64
notes: string
-- schema metadata --
huggingface: '{"info": {"features": {"path": {"dtype": "string", "_type":' + 179


Eval partition only, with the official eval protocol. Train and dev have to come from Edinburgh DataShare.

## 5. In-the-Wild

In [8]:
zf = zipfile.ZipFile(fs.open("datasets/mueller91/In-The-Wild/release_in_the_wild.zip"))
print(len(zf.namelist()), "entries")
pd.read_csv(zf.open("release_in_the_wild/meta.csv")).label.value_counts()

31782 entries


label
bona-fide    19963
spoof        11816
Name: count, dtype: int64

One zip with a `meta.csv` (file, speaker, label). Single files can be read from it without downloading the whole zip.